# Mapping SISAL cave sites from a local Turtle file

## About this notebook

This notebook reads a local RDF/Turtle snapshot of the **SISALv3 cave site
catalogue** (305 caves with geographic coordinates), parses it with `rdflib`,
and produces two interactive `folium` maps:

1. a marker map with every cave, colour-coded by archaeological status (plain
   speleothem site vs. confirmed archaeological cave site), and
2. a country-level choropleth that shows how the 305 SISAL caves distribute
   across the world's countries.

This is the **local Python companion** to the browser-executable
[`local_ttl-sisal-sites-live.qmd`](local_ttl-sisal-sites-live.qmd) notebook.
Both are structurally parallel — same query, same DataFrame schema, same two
maps — but the local variant uses the full scientific Python stack, whereas
the browser variant runs under Pyodide.

> **Source and attribution.** Data: **SISALv3** — [Kaushal et al. 2024,
> *Earth Syst. Sci. Data* 16, 1933–1963](https://doi.org/10.5194/essd-16-1933-2024) ·
> database DOI [10.5287/ora-2nanwp4rk](https://doi.org/10.5287/ora-2nanwp4rk).
> RDF conversion and enrichment (archaeological typing, UNESCO flags, Wikidata
> cross-links): the [GeoScience-FAIRification-LOD](https://github.com/Research-Squirrel-Engineers/GeoScience-FAIRification-LOD)
> repository by the Research Squirrel Engineers. The TTL snapshot used here
> was produced by
> [`plot_sisal_from_csv.py`](https://github.com/Research-Squirrel-Engineers/GeoScience-FAIRification-LOD/blob/main/SISAL/plot_sisal_from_csv.py)
> in that repository.

### Why this dataset?

SISAL (Speleothem Isotopes Synthesis and AnaLysis) is a global,
community-curated compilation of stable isotope records from cave carbonates.
Version 3 ships with a flat CSV of sites that has been converted to Linked
Open Data under the `geo-lod` vocabulary. As a teaching dataset it is
compact, globally distributed, and the archaeological enrichment layer
(cross-links to Wikidata and UNESCO World Heritage) makes it a nice bridge
between palaeoclimate and cultural-heritage data.

### What you'll learn

- How to parse a local TTL file with `rdflib` and run SPARQL queries against
  the in-memory graph
- How to turn SPARQL bindings into a tidy `pandas` DataFrame
- Two complementary cartographic idioms for site-level data: coloured
  markers (every cave individually, categorical colouring) and a country
  choropleth (aggregated count per country), both as interactive `folium`
  maps

### Data-context notes

- **Coordinate convention.** The TTL stores WKT literals as
  `<http://www.opengis.net/def/crs/EPSG/0/4326> POINT(lon lat)` — note the
  CRS prefix *and* the `lon lat` axis order mandated by GeoSPARQL. Leaflet
  (and `folium`) expect `[lat, lon]`, so we swap the order once on parse.
- **Archaeological sub-class.** A subset of sites is typed as
  `geolod:ArchaeologicalCaveSite` in addition to `geolod:Cave`. Seven sites
  are flagged as UNESCO World Heritage via `geolod:isUNESCOWorldHeritage`.
  We expose both categorical layers on the marker map.
- **Counts vs. samples.** `geolod:countD18OSamples` and
  `geolod:countD13CSamples` are per-site *observation* counts aggregated
  across all entities (speleothems) at that site. A single cave can host
  several speleothems; Corchia (site 145) has four. The country choropleth
  below counts *sites*, not observations.

### Requirements

```bash
pip install rdflib pandas folium requests shapely
```

`shapely` is used for the point-in-polygon country assignment; `requests`
fetches the Natural Earth country boundaries GeoJSON.

## 1  Setup, data loading, and SPARQL query

One cell handles everything from here to a clean DataFrame: parse the
graph, run one SPARQL query for every cave (label, coordinates, sample
counts, archaeological flags, and — where available — a Wikidata
identifier), and project the bindings into `pandas`.

In [ ]:
import re
import pandas as pd
from rdflib import Graph

g = Graph()
g.parse("sisal_sites.ttl", format="turtle")
print(f"Parsed {len(g):,} triples from sisal_sites.ttl")

SITES_QUERY = """
PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:   <http://www.w3.org/2002/07/owl#>
PREFIX geo:   <http://www.opengis.net/ont/geosparql#>
PREFIX geolod: <http://w3id.org/geo-lod/>

SELECT ?site ?label ?wkt ?nD18O ?nD13C ?isArch ?archCategory ?isUnesco ?wikidata
WHERE {
    ?site a geolod:Cave ;
          rdfs:label ?label ;
          geo:hasGeometry/geo:asWKT ?wkt ;
          geolod:countD18OSamples ?nD18O ;
          geolod:countD13CSamples ?nD13C .
    BIND(EXISTS { ?site a geolod:ArchaeologicalCaveSite } AS ?isArch)
    OPTIONAL { ?site geolod:archaeologicalCategory ?archCategory }
    BIND(EXISTS { ?site geolod:isUNESCOWorldHeritage true } AS ?isUnesco)
    OPTIONAL { ?site owl:sameAs ?wikidata
               FILTER(STRSTARTS(STR(?wikidata), "https://www.wikidata.org/")) }
}
ORDER BY ?site
"""

_WKT_RE = re.compile(r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)", re.IGNORECASE)

def parse_wkt(wkt):
    """Return (lat, lon) for a WKT Point literal, swapping WKT lon-lat order."""
    m = _WKT_RE.search(str(wkt) if wkt else "")
    if not m:
        return (None, None)
    try:
        return (float(m.group(2)), float(m.group(1)))
    except ValueError:
        return (None, None)

rows = []
for r in g.query(SITES_QUERY):
    lat, lon = parse_wkt(r["wkt"])
    rows.append({
        "site":       str(r["site"]).rsplit("/", 1)[-1],
        "label":      str(r["label"]),
        "lat":        lat,
        "lon":        lon,
        "nD18O":      int(r["nD18O"]),
        "nD13C":      int(r["nD13C"]),
        "isArch":     bool(r["isArch"]),
        "archCategory": str(r["archCategory"]) if r["archCategory"] else "",
        "isUnesco":   bool(r["isUnesco"]),
        "wikidata":   str(r["wikidata"]) if r["wikidata"] else "",
    })

df = pd.DataFrame(rows).dropna(subset=["lat", "lon"]).reset_index(drop=True)
print(f"Sites with coordinates: {len(df)}")
print(f"  Archaeological: {df['isArch'].sum()}")
print(f"  UNESCO World Heritage: {df['isUnesco'].sum()}")
print(f"  With Wikidata link: {(df['wikidata'] != '').sum()}")
print(f"  Total \u03b4\u00b9\u2078O observations: {df['nD18O'].sum():,}")
print(f"  Total \u03b4\u00b9\u00b3C observations: {df['nD13C'].sum():,}")
df.head()

## 2a  Marker map — every cave, typed by status

Each cave is plotted as a small circle; colour and layer group encode whether
it is a plain speleothem site, an archaeological cave site, or a UNESCO
World Heritage property. `folium`'s `LayerControl` doubles as a legend.

In [ ]:
import folium
from folium.features import DivIcon

def _category(row):
    if row["isUnesco"]:   return "UNESCO World Heritage"
    if row["isArch"]:     return "Archaeological cave"
    return "Speleothem site"

df["_category"] = df.apply(_category, axis=1)

CATEGORY_COLOURS = {
    "Speleothem site":       "#3388ff",
    "Archaeological cave":   "#e6550d",
    "UNESCO World Heritage": "#6a3d9a",
}
CATEGORY_ORDER = ["Speleothem site", "Archaeological cave", "UNESCO World Heritage"]

m_markers = folium.Map(location=[20, 10], zoom_start=2, world_copy_jump=True,
                       tiles="OpenStreetMap")
folium.TileLayer("Esri.WorldImagery", name="Esri World Imagery",
                 attr="Tiles &copy; Esri").add_to(m_markers)
folium.TileLayer("Esri.WorldTerrain", name="Esri World Terrain", max_zoom=13,
                 attr="Tiles &copy; Esri").add_to(m_markers)

# One FeatureGroup per category — each appears in the LayerControl with a swatch
category_groups = {}
for cat in CATEGORY_ORDER:
    sw = (f'<span style="display:inline-block;width:10px;height:10px;'
          f'background:{CATEGORY_COLOURS[cat]};border-radius:50%;'
          f'margin-right:6px;vertical-align:middle"></span>')
    category_groups[cat] = folium.FeatureGroup(name=sw + cat, show=True).add_to(m_markers)

for _, row in df.iterrows():
    cat = row["_category"]
    col = CATEGORY_COLOURS[cat]
    radius = {"UNESCO World Heritage": 8, "Archaeological cave": 7,
              "Speleothem site": 5}[cat]
    weight = 1 if cat == "Speleothem site" else 1.5

    popup_html = (f"<b>{row['label']}</b><br>{cat}<br>"
                  f"\u03b4\u00b9\u2078O: {int(row['nD18O']):,} \u00b7 "
                  f"\u03b4\u00b9\u00b3C: {int(row['nD13C']):,}")
    if row["archCategory"]:
        popup_html += f"<br><i>{row['archCategory']}</i>"
    if row["wikidata"]:
        popup_html += f'<br><a href="{row["wikidata"]}" target="_blank">Wikidata \u2197</a>'

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=radius, color=col, weight=weight,
        fill=True, fill_color=col, fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(category_groups[cat])

folium.LayerControl(collapsed=True).add_to(m_markers)
m_markers

## 2b  Country choropleth — sampling intensity per country

The aggregate view assigns each cave to a country (via point-in-polygon
against Natural Earth country boundaries) and colours each country by the
total number of SISAL caves it contains. Countries without SISAL sites are
drawn in a neutral grey; countries with sites get a sequential yellow → red
ramp. We do the point-in-polygon step in Python using `shapely`, then hand
the counts to `folium.GeoJson` for styling.

In [ ]:
import requests
from shapely.geometry import shape, Point
from shapely.strtree import STRtree

# Two mirrors \u2014 jsDelivr is usually fast, GitHub raw is the authoritative source.
GEOJSON_URLS = [
    ("https://cdn.jsdelivr.net/gh/nvkelso/natural-earth-vector"
     "@v5.1.2/geojson/ne_110m_admin_0_countries.geojson"),
    ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector"
     "/v5.1.2/geojson/ne_110m_admin_0_countries.geojson"),
]

def fetch_countries_geojson():
    last_error = None
    for url in GEOJSON_URLS:
        try:
            r = requests.get(url, timeout=30,
                             headers={"User-Agent": "SISAL-OER-notebook/1.0"})
            r.raise_for_status()
            return r.json(), url
        except Exception as e:
            last_error = e
            print(f"  \u26a0 {url} failed: {e}")
    raise RuntimeError(f"Could not fetch Natural Earth countries GeoJSON: {last_error}")

countries_geojson, used_url = fetch_countries_geojson()
print(f"Countries GeoJSON: {len(countries_geojson['features'])} features")
print(f"  from: {used_url}")

# Build a spatial index of country geometries for fast point-in-polygon
geoms = [shape(f["geometry"]) for f in countries_geojson["features"]]
tree  = STRtree(geoms)

# Count sites per country
site_counts = [0] * len(geoms)
for _, row in df.iterrows():
    pt = Point(row["lon"], row["lat"])
    for idx in tree.query(pt):
        if geoms[idx].contains(pt):
            site_counts[idx] += 1
            break

# Inject count into each feature's properties
for f, n in zip(countries_geojson["features"], site_counts):
    f["properties"]["_siteCount"] = n

max_n = max(site_counts) if site_counts else 1
n_with_sites = sum(1 for c in site_counts if c > 0)
print(f"Sites placed into countries: {sum(site_counts)}/{len(df)}")
print(f"Countries with at least one SISAL site: {n_with_sites}")
print(f"Max sites in a single country: {max_n}")

In [ ]:
# ── Style function: grey if zero, yellow→red ramp otherwise ──────
RAMP = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]

def ramp_colour(n, max_n=max_n, ramp=RAMP):
    if n <= 0:
        return "#e8e8e8"
    idx = min(len(ramp) - 1, int((n - 1) / max(max_n - 1, 1) * (len(ramp) - 1)))
    return ramp[idx]

def style_fn(feature):
    n = feature["properties"].get("_siteCount", 0)
    return {
        "fillColor": ramp_colour(n),
        "color":     "#666",
        "weight":    0.5,
        "fillOpacity": 0.75 if n > 0 else 0.35,
    }

m_choro = folium.Map(location=[20, 10], zoom_start=2, world_copy_jump=True,
                     tiles="OpenStreetMap")
folium.TileLayer("Esri.WorldTerrain", name="Esri World Terrain", max_zoom=13,
                 attr="Tiles &copy; Esri").add_to(m_choro)

folium.GeoJson(
    countries_geojson,
    name="SISAL sites by country",
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=["NAME", "_siteCount"],
        aliases=["Country:", "SISAL sites:"],
        localize=True, sticky=True,
    ),
).add_to(m_choro)

# ── Build a legend with integer count ranges ─────────────────────
step = max((max_n - 1) / len(RAMP), 1)
bins = []
for i, col in enumerate(RAMP):
    lo = int(1 + i * step)
    hi = int(1 + (i + 1) * step) - 1 if i < len(RAMP) - 1 else max_n
    bins.append((col, f"{lo}" if lo == hi else f"{lo}\u2013{hi}"))

legend_rows = (
    '<div><span style="display:inline-block;width:14px;height:14px;'
    'background:#e8e8e8;border:1px solid #666;opacity:0.6;'
    'margin-right:6px;vertical-align:middle"></span>none</div>'
)
for col, label in bins:
    legend_rows += (
        f'<div><span style="display:inline-block;width:14px;height:14px;'
        f'background:{col};border:1px solid #666;'
        f'margin-right:6px;vertical-align:middle"></span>{label}</div>'
    )

legend_html = f"""
<div style="position:fixed;bottom:20px;right:20px;z-index:9999;
            background:rgba(255,255,255,0.92);padding:8px 10px;
            border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,0.2);
            font-size:12px;line-height:1.5;font-family:sans-serif">
  <div style="font-weight:bold;margin-bottom:4px">Sites per country</div>
  {legend_rows}
  <div style="margin-top:4px;color:#666;font-size:11px">
    n = {len(df)} sites in {n_with_sites} countries
  </div>
</div>
"""
m_choro.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=True).add_to(m_choro)
m_choro

## 3  Explore

The full DataFrame is in scope — pick any dimension and aggregate or filter.
A few starting points:

In [ ]:
# Top 10 sites by total isotope observations
top = df.assign(nTotal=df["nD18O"] + df["nD13C"]) \
        .sort_values("nTotal", ascending=False)[
            ["label", "lat", "lon", "nD18O", "nD13C", "nTotal", "isArch", "isUnesco"]
        ].head(10)
top

In [ ]:
# Archaeological cave sites grouped by broader context
arch = df[df["isArch"]].copy()
print(f"Archaeological caves: {len(arch)}")
print(f"  with Wikidata link: {(arch['wikidata'] != '').sum()}")
print(f"  UNESCO World Heritage: {arch['isUnesco'].sum()}")
print()
print("Categories:")
print(arch["archCategory"].value_counts().to_string())

In [ ]:
# Which countries host the most SISAL sites?
country_counts = [
    (f["properties"].get("NAME", "?"), f["properties"]["_siteCount"])
    for f in countries_geojson["features"]
    if f["properties"]["_siteCount"] > 0
]
country_counts.sort(key=lambda kv: -kv[1])
print(f"{'Country':25s}  Sites")
print("-" * 36)
for name, n in country_counts[:15]:
    print(f"{name[:25]:25s}  {n:5d}")

In [ ]:
# Hier anpassen: filter or aggregate the DataFrame yourself
# Example — caves within Europe (bounding box)
europe = df[(df["lat"].between(35, 71)) & (df["lon"].between(-10, 40))]
print(f"European caves: {len(europe)}  "
      f"({len(europe)/len(df):.0%} of the catalogue)")

---

*Part of an Open Educational Resource series on knowledge graphs and linked
open data, produced in the context of [NFDI4Objects](https://www.nfdi4objects.net/).*
*Data source: [Kaushal et al. 2024, SISALv3](https://doi.org/10.5194/essd-16-1933-2024);*
*RDF conversion: [Research-Squirrel-Engineers/GeoScience-FAIRification-LOD](https://github.com/Research-Squirrel-Engineers/GeoScience-FAIRification-LOD).*